<img src="../assets/logo-banner.png" alt="radar-datatree — Cloud-native, time-aware weather radar datasets" width="800" class="only-light">
<img src="../assets/logo-banner-dark.png" alt="radar-datatree — Cloud-native, time-aware weather radar datasets" width="800" class="only-dark">

# Low sweeps, every VCP: assemble it, or open it pre-built
---

The lowest-elevation cut — `sweep_0` (≈ 0.5°), the one closest to the ground and the one rainfall estimation needs — is scattered across every Volume Coverage Pattern the radar has ever run. Stitching those into **one continuous, time-sorted series** spanning years is doable by hand, but it is the fiddly part. This notebook shows both ways to get there — assemble it yourself, or open a repo where it is already done — proves they return the same data, then uses the lowest sweep to plot a scan.

```{admonition} TL;DR
:class: important

**Grab a low sweep across the whole KLOT archive two ways** — glob every VCP and concatenate with `concat_sweep_across_vcps`, or open the pre-stitched `KLOT-lowsweeps` virtual repo in one line — then confirm they are byte-for-byte identical and plot a polarimetric snapshot.
```

```{admonition} Prerequisites
:class: note

This notebook builds on **[Notebook 1 — NEXRAD KLOT Demo](1.NEXRAD-KLOT-Demo)**:
- connecting to the public archive with `connect_to_nexrad_arco`
- opening a DataTree with `engine="rustytree"` and the `group_filter` glob
- flattening a sweep node with `to_dataset(inherit="all_coords")` and georeferencing it

If you are new to radar-datatree, start there first.
```

## Why the low sweep is scattered

A NEXRAD radar switches Volume Coverage Patterns as the weather changes — rapid-scan VCP-212 in severe convection, VCP-31 in clear air, and so on. Each VCP is a separate node in the DataTree with its **own** `vcp_time` axis, and each carries its own `sweep_0`. So the lowest-elevation record of the archive is spread across eight VCP nodes with no single timeline. To analyse it — accumulate rainfall, track a storm — you first have to knit those eight `sweep_0`s into one series ordered by time.

In [ ]:
# Colab bootstrap — runs only on Google Colab; no-op locally and in CI.
import sys

if "google.colab" in sys.modules:
    %pip install -qqq icechunk "rustytree-xarray>=0.3.0" "xradar>=0.12.0" \
        "zarr>=3.1.2" "s3fs>=2025.5.1" cmweather
    # Fetch the shared helper module so the imports below resolve on Colab.
    !wget -q https://raw.githubusercontent.com/AtmoScale/radar-datatree/main/notebooks/demo_functions.py

In [ ]:
import sys
from pathlib import Path

# Make notebooks/demo_functions.py importable when sphinx-build runs the
# rendered docs page from docs/ (a symlink to ../notebooks/). Harmless on Colab.
sys.path.insert(0, str(Path("../notebooks").resolve()))

import icechunk as ic
import numpy as np
import xarray as xr
import xradar  # noqa: F401  — registers the .xradar accessor
from demo_functions import concat_sweep_across_vcps, plot_polarimetric_panel

## Method A — glob every VCP and stitch it yourself

Open the native KLOT archive with an anonymous Icechunk session, then glob the lowest sweep from every VCP with `group_filter="/*/sweep_0"` — this returns eight separate nodes, each on its own timeline. We open them with `chunks={"vcp_time": 100}`: grouping 100 scans per chunk keeps the concatenate-and-sort step below tractable (the default one-scan-per-chunk layout would build a task graph over half a million chunks and crawl).

In [ ]:
# Anonymous, read-only session against the native KLOT archive.
storage = ic.s3_storage(
    bucket="nexrad-arco", prefix="KLOT", region="us-east-1", anonymous=True
)
session = ic.Repository.open(storage).readonly_session("main")

dt_glob = xr.open_datatree(
    session.store,
    engine="rustytree",
    group_filter="/*/sweep_0",
    chunks={"vcp_time": 100},
)
dt_glob

Now knit those eight `sweep_0` nodes into one series ordered by `vcp_time`. `concat_sweep_across_vcps` walks the `VCP-*` nodes, concatenates the chosen sweep along `vcp_time`, and sorts by time — the manual assembly, in one helper call.

In [ ]:
ds_glob = concat_sweep_across_vcps(dt_glob, sweep_name="sweep_0")
print(f"Assembled sweep_0: {ds_glob.sizes['vcp_time']:,} scans")
print(
    f"Time span: {str(ds_glob.vcp_time.values.min())[:19]} "
    f"-> {str(ds_glob.vcp_time.values.max())[:19]} UTC"
)

## Method B — open the pre-stitched archive

That assembly is the same for everyone, every time — so it has been done once, ahead of time. The **`KLOT-lowsweeps`** repo stores `sweep_0` and `sweep_1` already concatenated across all VCPs and sorted by time. It is a *virtual* repo — its chunks are references back into `s3://nexrad-arco/` — so reading its data needs that virtual-chunk container authorized (anonymously). With that one extra argument, opening it is otherwise identical.

In [ ]:
storage_low = ic.s3_storage(
    bucket="nexrad-arco", prefix="KLOT-lowsweeps", region="us-east-1", anonymous=True
)
repo_low = ic.Repository.open(
    storage_low,
    # Authorize anonymous reads of the virtual chunks that resolve to
    # s3://nexrad-arco/. Without this the open succeeds but the first
    # .compute() raises "authorize the virtual chunk container".
    authorize_virtual_chunk_access=ic.containers_credentials(
        {"s3://nexrad-arco/": ic.s3_anonymous_credentials()}
    ),
)
session_low = repo_low.readonly_session("main")

dt_low = xr.open_datatree(session_low.store, engine="rustytree", chunks={})
dt_low

In [ ]:
# The lowest sweep, already one continuous series — no glob, no concat.
ds_low = dt_low["sweep_0"].to_dataset(inherit="all_coords")
print(
    f"sweep_0: {ds_low.sizes['vcp_time']:,} scans "
    f"({str(ds_low.vcp_time.values.min())[:10]} -> {str(ds_low.vcp_time.values.max())[:10]})"
)
print(f"sweep_1 is here too: {dt_low['sweep_1'].sizes['vcp_time']:,} scans")

## Same data, either way

The virtual repo *is* the concatenation — precomputed. To prove it, take the same afternoon from both paths and compare the reflectivity value-for-value.

In [ ]:
window = ("2026-03-10 22:00", "2026-03-11 00:00")

a = ds_glob["DBZH"].sel(vcp_time=slice(*window)).compute()  # assembled by hand
b = ds_low["DBZH"].sel(vcp_time=slice(*window)).compute()  # pre-stitched

# Compare on the timestamps the two share.
shared = np.intersect1d(a.vcp_time.values, b.vcp_time.values)
a, b = a.sel(vcp_time=shared), b.sel(vcp_time=shared)
max_abs_diff = float(np.abs(np.nan_to_num(a.values) - np.nan_to_num(b.values)).max())

assert max_abs_diff == 0.0, f"paths disagree: max |Δ| = {max_abs_diff}"
print(f"{len(shared)} scans compared · max |Δ| DBZH = {max_abs_diff} — identical.")

## Plot a polarimetric snapshot

With the low sweep in hand, plotting is the easy part. Grab a single timestamp from the pre-stitched series, georeference it so the polar (azimuth, range) gates land on Cartesian (x, y) axes, and render the standard 2×2 polarimetric panel.

In [ ]:
# Snap to the scan closest to a March 2026 storm over Chicago.
scan = ds_low.sel(vcp_time="2026-03-10 23:20", method="nearest").xradar.georeference()
scan

In [ ]:
plot_polarimetric_panel(scan)

## Where to next

You now have the lowest sweep as one continuous time series. The analysis notebooks pick up from here:

**Reproduce a published figure?**
→ [Notebook 3 — QVP comparison](3.QVP-Workflow-Comparison) reproduces Ryzhkov et al. (2016) Fig. 4 and benchmarks ARCO streaming against the traditional file workflow.

**Estimate rainfall accumulation?**
→ [Notebook 4 — QPE scaling](4.QPE-Scaling-Benchmark) applies the Marshall–Palmer Z–R relation to exactly this low sweep, live for one day and scaling to seasons on a cluster.

---

← **[Notebook 1 — NEXRAD KLOT Demo](1.NEXRAD-KLOT-Demo)**

*Cite this work:* Ladino-Rincón, A., et al. (2026). *Radar DataTree: A FAIR and Cloud-Native Framework for Scalable Weather Radar Archives.* (Manuscript in preparation.) Earlier preprint: arXiv:2510.24943, [doi:10.48550/arXiv.2510.24943](https://doi.org/10.48550/arXiv.2510.24943).